# 02 - Pipeline Testing

Test the full extraction pipeline:
1. Deterministic-only mode (no API calls)
2. Full pipeline on a small subset
3. Full pipeline on all drugs
4. Inspect merged results

In [5]:
from pathlib import Path
import json

from livertox_extraction.pipeline import run_pipeline, load_all_results
from livertox_extraction.models import validate_extraction

XML_DIR = Path("/Users/maggiebrown/Desktop/axiom_bio/livertox-parser/data")
print(f"XML files: {len(list(XML_DIR.glob('*.xml')))}")

XML files: 85


## 1. Deterministic-only pipeline (no API calls)

In [13]:
results_det = run_pipeline(
    xml_dir=XML_DIR,
    output_dir="/Users/maggiebrown/Desktop/axiom_bio/livertox-parser/outputs/extractions_det_only",
    skip_llm=True,
)

Found 85 XML files to process
LLM extraction: DISABLED

[1/85] Processing Acetaminophen...
  Done: 3 fields extracted
[2/85] Processing Ambrisentan...
  Done: 1 fields extracted
[3/85] Processing Amiodarone...
  Done: 4 fields extracted
[4/85] Processing Amitriptyline...
  Done: 4 fields extracted
[5/85] Processing Amodiaquine...
  Done: 1 fields extracted
[6/85] Processing Amoxicillin...
  Done: 4 fields extracted
[7/85] Processing Atorvastatin...
  Done: 5 fields extracted
[8/85] Processing Azathioprine...
  Done: 3 fields extracted
[9/85] Processing Benzbromarone...
  Done: 2 fields extracted
[10/85] Processing Benztropine...
  Done: 1 fields extracted
[11/85] Processing Bosentan...
  Done: 4 fields extracted
[12/85] Processing Buspirone...
  Done: 1 fields extracted
[13/85] Processing Carbamazepine...
  Done: 3 fields extracted
[14/85] Processing Cardizepam...
  Done: 0 fields extracted
[15/85] Processing Celecoxib...
  Done: 3 fields extracted
[16/85] Processing Chlorpromazine...


In [14]:
# Summarize deterministic results
print(f"{'Drug':<20} {'DILI':>5} {'Pattern':<16} {'R':>6} {'Enz%':>7} {'Fields':>6}")
print("-" * 65)

for name, result in sorted(results_det.items()):
    d = result.to_dict()
    filled = sum(1 for k, v in d.items() if k != "drug_name" and v is not None and v is not False)
    print(f"{name:<20} {str(result.dili_likelihood_score or '-'):>5} "
          f"{str(result.injury_pattern or '-'):<16} "
          f"{str(result.r_ratio or '-'):>6} "
          f"{str(result.fraction_patients_with_enzyme_elevation or '-'):>7} "
          f"{filled:>6}")

Drug                  DILI Pattern               R    Enz% Fields
-----------------------------------------------------------------
Acetaminophen            A hepatocellular     14.0       -      3
Ambrisentan              E -                     -       -      1
Amiodarone               A hepatocellular     13.1     0.5      4
Amitriptyline            B hepatocellular     22.0    0.12      4
Amodiaquine              A -                     -       -      1
Amoxicillin              B cholestatic         1.2       -      4
Atorvastatin             A cholestatic         3.0    0.03      5
Azathioprine             A mixed               0.9       -      3
Benzbromarone            B -                     -   0.001      2
Benztropine              E -                     -       -      1
Bosentan                 C mixed               2.5    0.04      4
Buspirone                E -                     -       -      1
Carbamazepine            A mixed               4.6       -      3
Cardizepam

## 2. Full pipeline on a small subset (uses API)

In [15]:
# Test on 3 drugs: one rich, one minimal, one fictional
results_small = run_pipeline(
    xml_dir=XML_DIR,
    output_dir="/Users/maggiebrown/Desktop/axiom_bio/livertox-parser/outputs/extractions_test",
    keys_file="/Users/maggiebrown/Desktop/axiom_bio/livertox-parser/keys.txt",
    drug_names=["Zileuton", "Benztropine", "Gastrozine"],
)

Found 3 XML files to process
LLM extraction: ENABLED

Anthropic client created successfully

[1/3] Processing Benztropine...
  Done: 2 fields extracted
[2/3] Processing Gastrozine...
  Done: 0 fields extracted
[3/3] Processing Zileuton...
  Done: 9 fields extracted

PIPELINE SUMMARY
Total drugs: 3
Successfully parsed: 3
Parse failures: 0 []
LLM failures: 0 []
Validation warnings: 0
Time elapsed: 6.0 seconds
Results saved to: /Users/maggiebrown/Desktop/axiom_bio/livertox-parser/outputs/extractions_test



In [16]:
# Inspect merged results
for name, result in results_small.items():
    print(f"=== {name} ===")
    print(json.dumps(result.to_dict(), indent=2))
    print(f"Validation: {validate_extraction(result)}")
    print()

=== Benztropine ===
{
  "drug_name": "Benztropine",
  "dili_likelihood_score": "E",
  "injury_pattern": null,
  "fraction_patients_with_enzyme_elevation": null,
  "fraction_patients_with_dili": null,
  "is_immune_mediated": false,
  "risk_factors": null,
  "safe_dose": {
    "value": 0.5,
    "unit": "mg",
    "frequency": "daily"
  },
  "toxic_dose": null,
  "onset_time": null,
  "peak_alt": null,
  "peak_alp": null,
  "r_ratio": null,
  "bilirubin_peak": null,
  "regulatory_status": null
}
Validation: []

=== Gastrozine ===
{
  "drug_name": "Gastrozine",
  "dili_likelihood_score": null,
  "injury_pattern": null,
  "fraction_patients_with_enzyme_elevation": null,
  "fraction_patients_with_dili": null,
  "is_immune_mediated": false,
  "risk_factors": null,
  "safe_dose": null,
  "toxic_dose": null,
  "onset_time": null,
  "peak_alt": null,
  "peak_alp": null,
  "r_ratio": null,
  "bilirubin_peak": null,
  "regulatory_status": null
}
Validation: []

=== Zileuton ===
{
  "drug_name": "Zi

## 3. Full pipeline on ALL drugs (takes a few minutes)

In [19]:
results_all = run_pipeline(
    xml_dir=XML_DIR,
    output_dir="/Users/maggiebrown/Desktop/axiom_bio/livertox-parser/outputs/extractions",
    keys_file="/Users/maggiebrown/Desktop/axiom_bio/livertox-parser/keys.txt",
)

Found 85 XML files to process
LLM extraction: ENABLED

Anthropic client created successfully

[1/85] Processing Acetaminophen...
  Done: 8 fields extracted
[2/85] Processing Ambrisentan...
  Done: 3 fields extracted
[3/85] Processing Amiodarone...
  Done: 9 fields extracted
[4/85] Processing Amitriptyline...
  Done: 7 fields extracted
[5/85] Processing Amodiaquine...
  Done: 6 fields extracted
[6/85] Processing Amoxicillin...
  Done: 7 fields extracted
[7/85] Processing Atorvastatin...
  Done: 10 fields extracted
[8/85] Processing Azathioprine...
  Done: 7 fields extracted
[9/85] Processing Benzbromarone...
  Done: 4 fields extracted
[10/85] Processing Benztropine...
  Done: 2 fields extracted
[11/85] Processing Bosentan...
  Done: 6 fields extracted
[12/85] Processing Buspirone...
  Done: 2 fields extracted
[13/85] Processing Carbamazepine...
  Done: 7 fields extracted
[14/85] Processing Cardizepam...
  Done: 0 fields extracted
[15/85] Processing Celecoxib...
  Done: 8 fields extracte

## 4. Load and inspect saved results

In [20]:
# After running the full pipeline, load results from disk
results_loaded = load_all_results("/Users/maggiebrown/Desktop/axiom_bio/livertox-parser/outputs/extractions")
print(f"Loaded {len(results_loaded)} results")

Loaded 85 results


In [21]:
# Compare deterministic vs merged field counts
for name in sorted(results_all.keys()):
    det_d = results_det.get(name)
    full_d = results_all.get(name)
    if det_d and full_d:
        det_filled = sum(1 for k, v in det_d.to_dict().items() if k != "drug_name" and v is not None and v is not False)
        full_filled = sum(1 for k, v in full_d.to_dict().items() if k != "drug_name" and v is not None and v is not False)
        gained = full_filled - det_filled
        if gained > 0:
            print(f"{name:<20} det={det_filled}  full={full_filled}  LLM added {gained} fields")

Acetaminophen        det=3  full=8  LLM added 5 fields
Ambrisentan          det=1  full=3  LLM added 2 fields
Amiodarone           det=4  full=9  LLM added 5 fields
Amitriptyline        det=4  full=7  LLM added 3 fields
Amodiaquine          det=1  full=6  LLM added 5 fields
Amoxicillin          det=4  full=7  LLM added 3 fields
Atorvastatin         det=5  full=10  LLM added 5 fields
Azathioprine         det=3  full=7  LLM added 4 fields
Benzbromarone        det=2  full=4  LLM added 2 fields
Benztropine          det=1  full=2  LLM added 1 fields
Bosentan             det=4  full=6  LLM added 2 fields
Buspirone            det=1  full=2  LLM added 1 fields
Carbamazepine        det=3  full=7  LLM added 4 fields
Celecoxib            det=3  full=8  LLM added 5 fields
Chlorpromazine       det=3  full=7  LLM added 4 fields
Clomipramine         det=2  full=5  LLM added 3 fields
Clozapine            det=3  full=8  LLM added 5 fields
Cyclophosphamide     det=4  full=7  LLM added 3 fields
Dabigatra

## Pipeline Results Summary

### Run Statistics
- **Total XML files**: 85
- **Successfully parsed**: 84 (HepC.xml failed — malformed XML)
- **LLM extraction failures**: 0
- **Validation warnings**: 0
- **Full pipeline runtime**: ~3-4 minutes for 85 drugs

### Deterministic vs Full Pipeline Field Counts
| Metric | Deterministic Only | Full (Det + LLM) |
|--------|-------------------|-------------------|
| Avg fields per real drug | ~2.8 | ~6.3 |
| Max fields extracted | 7 (Zileuton) | 10 (Atorvastatin, Methyldopa) |
| Drugs with 0 fields | 12 (fictional + HepC) | 12 (same — correctly unchanged) |
| LLM fields added per drug | — | 1-5 typically |

### Key Observations

**Merge strategy is working correctly:**
- Deterministic values (DILI score, R-ratio, injury pattern) are preserved
- LLM fills in semantic fields (safe_dose, onset_time, risk_factors, is_immune_mediated)
- Example: Zileuton det=7 → full=9, gained safe_dose and onset_time from LLM

**Fictional drugs handled correctly:**
- Cardizepam, Dermacillin, Endocrinex, Gastrozine, Hematolix, Hepatrofin, Immunexor, Neuroxalin, Pulmoxate, Renalyte: all 0 fields, no hallucination

**Edge cases:**
- Flavoxate: 0 fields deterministic → 1 field with LLM (has hepatotoxicity text but no standard likelihood score format)
- Nicardipine: 0 fields deterministic → 2 fields with LLM (same issue)
- Felodipine and Felodipinee: identical results (3 fields each), confirming Felodipinee is a duplicate

### Next Steps
- Build gold standard annotations for ~15 drugs
- Build evaluation framework (evaluate.py) comparing pipeline output to gold standard
- Generate HTML report (report.py) with charts and metrics